In [1]:
# ================================================
# ISY503 - Intelligent Systems - Assessment 3
# Person 1 - Data & Preprocessing Lead
# ================================================

# This uploads your .tar file from your computer into Colab
# A "Choose Files" button will appear when you run this
# Click it and select "domain_sentiment_data.tar" from your Desktop/Tashi Deki_ISY503 folder

from google.colab import files

print("👇 Click 'Choose Files' and select domain_sentiment_data.tar")
uploaded = files.upload()
print("✅ Upload done!")

👇 Click 'Choose Files' and select domain_sentiment_data.tar


Saving domain_sentiment_data.tar.gz to domain_sentiment_data.tar.gz
✅ Upload done!


In [4]:
# This extracts the .tar file so we can access the review folders inside
# .tar is like a zip file - we need to "open" it to get the folders out

import tarfile
import os

print("Extracting tar.gz file...")

with tarfile.open("domain_sentiment_data.tar.gz", "r:gz") as tar:
    tar.extractall("data")

print("✅ Extracted successfully!")
print("\nFolders found inside:")
print(os.listdir("data/sorted_data_acl"))

Extracting tar.gz file...


/tmp/ipykernel_6111/2856218488.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("data")


✅ Extracted successfully!

Folders found inside:
['electronics', 'books', 'dvd', 'kitchen_&_housewares']


In [5]:
# ================================================
# STEP 1: INSTALL AND IMPORT ALL LIBRARIES
# ================================================

# These are the tools we need for preprocessing
# numpy    → handles large lists of numbers efficiently
# sklearn  → has the train/test split tool
# keras    → has the tokenizer and padding tools

import os
import re
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("✅ All libraries loaded successfully!")

✅ All libraries loaded successfully!


In [6]:
# ================================================
# STEP 2: CONFIGURATION SETTINGS
# ================================================

# These are the control settings for our preprocessing
# If we want to change anything later, we only change it here

DATA_DIR = "data/sorted_data_acl"  # 📁 Where our dataset folders are
MAX_VOCAB_SIZE = 10000              # Only use the 10,000 most common words
MAX_SEQUENCE_LENGTH = 200           # Every review will be exactly 200 words long
MIN_REVIEW_LENGTH = 10              # Throw away reviews shorter than 10 words
TEST_SIZE = 0.1                     # 10% of data saved for final testing
VAL_SIZE = 0.1                      # 10% of data saved for validation
RANDOM_SEED = 42                    # Makes results the same every time we run

print("✅ Settings configured!")
print(f"   Data folder     : {DATA_DIR}")
print(f"   Vocabulary size : {MAX_VOCAB_SIZE} words")
print(f"   Review length   : {MAX_SEQUENCE_LENGTH} words")
print(f"   Min length      : {MIN_REVIEW_LENGTH} words")
print(f"   Train/Val/Test  : 80% / 10% / 10%")

✅ Settings configured!
   Data folder     : data/sorted_data_acl
   Vocabulary size : 10000 words
   Review length   : 200 words
   Min length      : 10 words
   Train/Val/Test  : 80% / 10% / 10%


In [11]:
# ================================================
# STEP 3: LOAD ALL REVIEWS FROM THE DATASET
# Assignment requirement: "Load all negative and positive comments"
# ================================================

def parse_reviews(filepath):
    """
    Opens ONE review file and pulls out just the review text.

    Inside each file, reviews are stored like this:
    <review_text>
        This product is amazing!
    </review_text>

    This function finds everything between those tags
    and returns a list of review texts.
    """
    reviews = []

    # Open and read the whole file
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # re.findall searches for a pattern in text
    # Here it finds everything between <review_text> and </review_text>
    matches = re.findall(r'<review_text>(.*?)</review_text>', content, re.DOTALL)

    for match in matches:
        reviews.append(match.strip())  # .strip() removes blank lines

    return reviews


def load_all_reviews(data_dir):
    """
    Goes through EVERY category folder:
    books, dvd, electronics, kitchen_&_housewares

    In each folder it loads:
    - positive.review file → goes into all_positive list
    - negative.review file → goes into all_negative list
    """
    all_positive = []
    all_negative = []

    for category in os.listdir(data_dir):
        category_path = os.path.join(data_dir, category)

        # Skip if it's not a folder
        if not os.path.isdir(category_path):
            continue

        # Build path to positive and negative files
        pos_file = os.path.join(category_path, 'positive.review')
        neg_file = os.path.join(category_path, 'negative.review')

        # Load positive reviews
        if os.path.exists(pos_file):
            pos_reviews = parse_reviews(pos_file)
            all_positive.extend(pos_reviews)
            print(f"  📗 {len(pos_reviews):4d} positive reviews from '{category}'")

        # Load negative reviews
        if os.path.exists(neg_file):
            neg_reviews = parse_reviews(neg_file)
            all_negative.extend(neg_reviews)
            print(f"  📕 {len(neg_reviews):4d} negative reviews from '{category}'")

    return all_positive, all_negative


# ── RUN IT ──
print("🔵 Loading all reviews...\n")
positive_reviews, negative_reviews = load_all_reviews(DATA_DIR)

print(f"\n{'='*40}")
print(f"✅ Total POSITIVE reviews: {len(positive_reviews)}")
print(f"✅ Total NEGATIVE reviews: {len(negative_reviews)}")
print(f"✅ Total overall        : {len(positive_reviews) + len(negative_reviews)}")

🔵 Loading all reviews...

  📗 1000 positive reviews from 'electronics'
  📕 1000 negative reviews from 'electronics'
  📗 1000 positive reviews from 'books'
  📕 1000 negative reviews from 'books'
  📗 1000 positive reviews from 'dvd'
  📕 1000 negative reviews from 'dvd'
  📗 1000 positive reviews from 'kitchen_&_housewares'
  📕 1000 negative reviews from 'kitchen_&_housewares'

✅ Total POSITIVE reviews: 4000
✅ Total NEGATIVE reviews: 4000
✅ Total overall        : 8000


In [12]:
# ================================================
# STEP 4: COMBINE REVIEWS AND CREATE LABELS
# Assignment requirement: "mix and randomise the data"
# ================================================

# Put all reviews into ONE list
all_reviews = positive_reviews + negative_reviews

# Create matching labels
# 1 = Positive review
# 0 = Negative review
all_labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

print("✅ Reviews combined!")
print(f"   Total reviews : {len(all_reviews)}")
print(f"   Total labels  : {len(all_labels)}")
print(f"\n   First label  : {all_labels[0]}  (1 = positive)")
print(f"   Last label   : {all_labels[-1]} (0 = negative)")
print(f"\n   Example review:")
print(f"   '{all_reviews[0][:100]}...'")

✅ Reviews combined!
   Total reviews : 8000
   Total labels  : 8000

   First label  : 1  (1 = positive)
   Last label   : 0 (0 = negative)

   Example review:
   'I purchased this unit due to frequent blackouts in my area and 2 power supplies going bad.  It will ...'


In [13]:
# ================================================
# STEP 5: CLEAN THE TEXT
# Assignment requirement: "Clean the data (punctuation, spelling etc.)"
# ================================================

def clean_text(text):
    """
    Cleans a single review:

    BEFORE: "This product!! Is GREAT 10/10 :)"
    AFTER:  "this product is great"

    3 things happen:
    1. Everything becomes lowercase
    2. Punctuation and numbers are removed
    3. Extra spaces are removed
    """
    # Step 1: Make everything lowercase
    # "GREAT" and "great" should be treated as the same word
    text = text.lower()

    # Step 2: Remove anything that is NOT a letter or space
    # [^a-z\s] means "not a letter from a-z and not a space"
    # This removes: !, ?, ., ,, 1, 2, 3, :, ), (, etc.
    text = re.sub(r'[^a-z\s]', '', text)

    # Step 3: Remove extra blank spaces
    # \s+ means "one or more spaces"
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply clean_text to EVERY review in our list
# This is called a "list comprehension" - it's a fast way to loop
print("🔵 Cleaning all reviews...\n")
all_reviews_clean = [clean_text(r) for r in all_reviews]

# Show before and after example
print("BEFORE cleaning:")
print(f"  '{all_reviews[0][:150]}'")
print("\nAFTER cleaning:")
print(f"  '{all_reviews_clean[0][:150]}'")

# Replace original with cleaned version
all_reviews = all_reviews_clean
print(f"\n✅ All {len(all_reviews)} reviews cleaned!")

🔵 Cleaning all reviews...

BEFORE cleaning:
  'I purchased this unit due to frequent blackouts in my area and 2 power supplies going bad.  It will run my cable modem, router, PC, and LCD monitor fo'

AFTER cleaning:
  'i purchased this unit due to frequent blackouts in my area and power supplies going bad it will run my cable modem router pc and lcd monitor for minut'

✅ All 8000 reviews cleaned!


In [14]:
# ================================================
# STEP 6: REMOVE OUTLIERS
# Assignment requirement: "Conduct outlier removal to
#                          eliminate really short or wrong reviews"
# ================================================

def remove_outliers(reviews, labels, min_length=MIN_REVIEW_LENGTH):
    """
    Removes reviews that are too short to be useful.

    Why do we do this?
    A review like "ok" or "bad product" only has 1-2 words.
    The AI cannot learn any useful patterns from such short text.
    We need at least 10 words to detect sentiment properly.

    ETHICAL NOTE for your report:
    Removing short reviews is a decision that could introduce bias.
    Some people naturally write shorter reviews. We document this
    choice so it is transparent.
    """
    good_reviews = []
    good_labels  = []
    removed      = 0

    for review, label in zip(reviews, labels):
        # Count how many words are in this review
        word_count = len(review.split())

        if word_count >= min_length:
            # Keep this review - it's long enough
            good_reviews.append(review)
            good_labels.append(label)
        else:
            # Throw away this review - too short
            removed += 1

    return good_reviews, good_labels, removed


# ── RUN IT ──
print("🔵 Removing outliers (reviews under 10 words)...\n")

all_reviews, all_labels, removed_count = remove_outliers(all_reviews, all_labels)

print(f"  🗑️  Removed  : {removed_count} reviews (too short)")
print(f"  ✅ Remaining : {len(all_reviews)} reviews")
print(f"\n  Positive remaining: {sum(all_labels)}")
print(f"  Negative remaining: {len(all_labels) - sum(all_labels)}")

🔵 Removing outliers (reviews under 10 words)...

  🗑️  Removed  : 24 reviews (too short)
  ✅ Remaining : 7976 reviews

  Positive remaining: 3987
  Negative remaining: 3989


In [15]:
# ================================================
# STEP 7: TOKENISATION
# Assignment requirement: "Encode the words in the review"
# ================================================

def tokenise_reviews(reviews, max_vocab=MAX_VOCAB_SIZE):
    """
    Converts every word into a unique number.

    Why? Computers cannot understand English words.
    They only understand numbers. So we build a dictionary:

    "the"      → 1
    "product"  → 2
    "is"       → 3
    "amazing"  → 4

    So the review "the product is amazing" becomes [1, 2, 3, 4]

    oov_token='<OOV>' handles unknown words.
    OOV = Out Of Vocabulary (words the model has never seen)
    """
    # Create the tokenizer tool
    tokenizer = Tokenizer(num_words=max_vocab, oov_token='<OOV>')

    # Read ALL reviews to learn what words exist
    tokenizer.fit_on_texts(reviews)

    # Convert every review from words → numbers
    sequences = tokenizer.texts_to_sequences(reviews)

    return sequences, tokenizer


# ── RUN IT ──
print("🔵 Tokenising reviews (converting words to numbers)...\n")

sequences, tokenizer = tokenise_reviews(all_reviews)

print(f"  ✅ Total unique words found : {len(tokenizer.word_index)}")
print(f"  ✅ Using top               : {MAX_VOCAB_SIZE} words")
print(f"\n  Example:")
print(f"  Original text : '{all_reviews[0][:60]}'")
print(f"  As numbers    : {sequences[0][:12]}...")

🔵 Tokenising reviews (converting words to numbers)...

  ✅ Total unique words found : 44827
  ✅ Using top               : 10000 words

  Example:
  Original text : 'i purchased this unit due to frequent blackouts in my area a'
  As numbers    : [7, 269, 10, 210, 578, 4, 4163, 1, 11, 23, 1046, 3]...


In [16]:
# ================================================
# STEP 8: ENCODE THE LABELS
# Assignment requirement: "Encode the labels for positive and negative"
# ================================================

# Convert our Python list [1, 0, 1, 0, ...]
# into a numpy array — a fast number format that Keras needs

y = np.array(all_labels)

print("🔵 Encoding labels...\n")
print(f"  ✅ Labels converted to numpy array")
print(f"  Total labels    : {len(y)}")
print(f"  Positive (1s)   : {np.sum(y == 1)}")
print(f"  Negative (0s)   : {np.sum(y == 0)}")
print(f"  First 10 labels : {y[:10]}")
print(f"\n  Shape of y: {y.shape}")
print("  → This means we have", y.shape[0], "labels total")

🔵 Encoding labels...

  ✅ Labels converted to numpy array
  Total labels    : 7976
  Positive (1s)   : 3987
  Negative (0s)   : 3989
  First 10 labels : [1 1 1 1 1 1 1 1 1 1]

  Shape of y: (7976,)
  → This means we have 7976 labels total


In [17]:
# ================================================
# STEP 9: PADDING
# Assignment requirement: "Pad/truncate remaining data"
# ================================================

print("🔵 Padding sequences...\n")

# pad_sequences makes ALL reviews exactly the same length
#
# Why do we need this?
# The AI model needs all inputs to be exactly the same size
# Just like a form where every box must be filled
#
# Short review [1, 2, 3, 4] becomes:
# [1, 2, 3, 4, 0, 0, 0, 0, 0, ... 0]  ← zeros added at END
#
# Long review (over 200 words) gets cut off at 200 words

X = pad_sequences(
    sequences,
    maxlen=MAX_SEQUENCE_LENGTH,  # exactly 200 numbers long
    padding='post',              # add zeros at the END
    truncating='post'            # cut from the END if too long
)

print(f"  ✅ Padding complete!")
print(f"  Shape of X: {X.shape}")
print(f"  → {X.shape[0]} reviews, each exactly {X.shape[1]} numbers long")
print(f"\n  Example (first 20 numbers of review 1):")
print(f"  {X[0][:20]}")
print(f"\n  Notice the zeros at the end = padding")
print(f"  Last 10 numbers: {X[0][-10:]}")

🔵 Padding sequences...

  ✅ Padding complete!
  Shape of X: (7976, 200)
  → 7976 reviews, each exactly 200 numbers long

  Example (first 20 numbers of review 1):
  [   7  269   10  210  578    4 4163    1   11   23 1046    3  308 9060
  161  148    9   54  548   23]

  Notice the zeros at the end = padding
  Last 10 numbers: [0 0 0 0 0 0 0 0 0 0]


In [18]:
# ================================================
# STEP 10: TRAIN / VALIDATION / TEST SPLIT
# Assignment requirement: "Split the data into training,
#                          validation and test sets"
# ================================================

print("🔵 Splitting data into train/validation/test sets...\n")

# WHY do we split?
# We cannot test the AI on data it already learned from
# That would be like giving a student the exam answers to study
# then marking them on the exact same exam - it would be cheating!
#
# So we hide some data from the AI during training:
# TRAINING   (80%) → AI learns from this
# VALIDATION (10%) → checked during training to avoid overfitting
# TESTING    (10%) → only used at the very end for final accuracy

# STEP A: Take out 10% for testing
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,       # 10% goes to test set
    random_state=RANDOM_SEED,  # same split every time we run
    stratify=y                 # keeps balanced positive/negative in each split
)

# STEP B: Split remaining 90% into 80% train + 10% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_SEED,
    stratify=y_temp
)

total = len(X_train) + len(X_val) + len(X_test)

print(f"  ✅ Split complete!")
print(f"  {'Set':<12} {'Samples':>8} {'Percentage':>12}")
print(f"  {'-'*35}")
print(f"  {'Training':<12} {len(X_train):>8} {len(X_train)/total*100:>11.0f}%")
print(f"  {'Validation':<12} {len(X_val):>8} {len(X_val)/total*100:>11.0f}%")
print(f"  {'Testing':<12} {len(X_test):>8} {len(X_test)/total*100:>11.0f}%")
print(f"  {'-'*35}")
print(f"  {'Total':<12} {total:>8} {'100%':>12}")
print(f"\n  Shape of X_train: {X_train.shape}")
print(f"  Shape of X_val  : {X_val.shape}")
print(f"  Shape of X_test : {X_test.shape}")

🔵 Splitting data into train/validation/test sets...

  ✅ Split complete!
  Set           Samples   Percentage
  -----------------------------------
  Training         6380          80%
  Validation        798          10%
  Testing           798          10%
  -----------------------------------
  Total            7976         100%

  Shape of X_train: (6380, 200)
  Shape of X_val  : (798, 200)
  Shape of X_test : (798, 200)


In [19]:
# ================================================
# STEP 11: SAVE ALL FILES
# These files get shared with Person 2 and Person 3
# ================================================

import pickle

print("💾 Saving all processed files...\n")

# Save the review data as numpy files (.npy)
# These contain the padded number sequences
np.save('X_train.npy', X_train)  # training reviews → Person 2
np.save('X_val.npy',   X_val)    # validation reviews → Person 2
np.save('X_test.npy',  X_test)   # test reviews → Person 2

# Save the labels (0s and 1s)
np.save('y_train.npy', y_train)  # training labels → Person 2
np.save('y_val.npy',   y_val)    # validation labels → Person 2
np.save('y_test.npy',  y_test)   # test labels → Person 2

# Save the tokenizer
# Person 3 NEEDS this for the web app
# It knows how to convert NEW text typed by users into numbers
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("  ✅ X_train.npy saved  → send to Person 2")
print("  ✅ X_val.npy   saved  → send to Person 2")
print("  ✅ X_test.npy  saved  → send to Person 2")
print("  ✅ y_train.npy saved  → send to Person 2")
print("  ✅ y_val.npy   saved  → send to Person 2")
print("  ✅ y_test.npy  saved  → send to Person 2")
print("  ✅ tokenizer.pkl saved → send to Person 3")
print("\n🟢 ALL PREPROCESSING DONE!")

💾 Saving all processed files...

  ✅ X_train.npy saved  → send to Person 2
  ✅ X_val.npy   saved  → send to Person 2
  ✅ X_test.npy  saved  → send to Person 2
  ✅ y_train.npy saved  → send to Person 2
  ✅ y_val.npy   saved  → send to Person 2
  ✅ y_test.npy  saved  → send to Person 2
  ✅ tokenizer.pkl saved → send to Person 3

🟢 ALL PREPROCESSING DONE!


In [20]:
# ================================================
# STEP 12: DOWNLOAD ALL FILES TO YOUR COMPUTER
# ================================================

from google.colab import files

print("📥 Downloading files to your computer...\n")
print("Each file will download one by one")
print("Check your Downloads folder after!\n")

files.download('X_train.npy')
files.download('X_val.npy')
files.download('X_test.npy')
files.download('y_train.npy')
files.download('y_val.npy')
files.download('y_test.npy')
files.download('tokenizer.pkl')

print("✅ All 7 files downloaded!")
print("Now upload them to GitHub to share with teammates")

📥 Downloading files to your computer...

Each file will download one by one
Check your Downloads folder after!



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All 7 files downloaded!
Now upload them to GitHub to share with teammates
